# nb_VHybrid_Transforms

**Purpose:** Companion to `df_StagingPassthrough` (Dataflow Gen2). Reads the `stg_*_Simple` tables that the dataflow has already populated with Direct / Default / single-column-Crosswalk values, then layers on the rules that are too complex for Power Query:

- multi-source joins (3+ tables)
- string parsing (`NAM`, `HEIGHT`)
- conditional FIPS / relationship logic
- windowed sequence numbers
- aggregated balance buckets

Produces the final `VDEMO`, `VCASE`, `VCMEM`, `VSORD`, `VOBLE`, `VLSUP` tables in `TargetLH`.

> The **stand-alone, code-only** version of this work is `DACSES_Fabric_Transformations.ipynb` at the project root. Use that one when running outside the hybrid pipeline.

## 1. Parameters

In [ ]:
# Injected by pl_DACSES_Convert at runtime
SOURCE_LAKEHOUSE = "SourceLH"
TARGET_LAKEHOUSE = "TargetLH"
CONVERSION_DATE  = "2026-05-06"
DELAWARE_FIPS    = "1000000"

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType, DateType

def src(name):  return spark.read.table(f"{SOURCE_LAKEHOUSE}.{name}")
def stg(name):  return spark.read.table(f"{TARGET_LAKEHOUSE}.{name}")

# Captured row counts per target table — returned to the pipeline at the end
# via notebookutils.notebook.exit so Validate_Row_Counts can compare against
# the SQL Server load.
ROW_COUNTS = {}

def write_target(df, name):
    fq = f"{TARGET_LAKEHOUSE}.{name}"
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(fq))
    # Read the count from the written Delta table — avoids re-running the
    # entire transformation DAG just to log a count.
    n = spark.read.table(fq).count()
    ROW_COUNTS[name] = n
    print(f"{name:<6} -> {n:>5} rows")

# Target SQL Server 2022 type alignment (see TargetDB/DACESS_Target_DDL.sql).
# The pipeline bulk-loads these Delta tables into DACSES_POC_Target via Copy
# activities; cast keys/amounts to the exact precisions to avoid TDS overflow.
TARGET_TYPES = {
    "Case_IDNO":          DecimalType(6, 0),
    "MemberMci_IDNO":     DecimalType(10, 0),
    "INDIVIDUAL_IDNO":    DecimalType(8, 0),
    "MemberSsn_NUMB":     DecimalType(9, 0),
    "OrderSeq_NUMB":      DecimalType(2, 0),
    "ObligationSeq_NUMB": DecimalType(2, 0),
    "Order_IDNO":         DecimalType(15, 0),
}
AMOUNT_PRECISION = DecimalType(11, 2)

def conform(df):
    """Cast key/amount columns to target SQL Server precisions before bulk load."""
    for c, t in TARGET_TYPES.items():
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(t))
    for c in df.columns:
        if c.endswith("_AMNT"):
            df = df.withColumn(c, F.col(c).cast(AMOUNT_PRECISION))
    return df

## 2. VDEMO  –  finish what the dataflow started

**From dataflow:** `MemberMci_IDNO`, `INDIVIDUAL_IDNO`, `FullDisplay_NAME`, `MemberSex_CODE`, `MemberSsn_NUMB`, `Birth_DATE`, `Race_CODE`, defaults.

**Adds here:** parsed Last/First/Middle, height/weight, BirthCity/State, driver license, mail-address phone.

In [ ]:
stg_vdemo = stg("stg_VDEMO_Simple")
T1045 = src("T1045_AP_EXT")
T1005 = src("T1005_LS_LIC_MATCH")
T1051 = src("T1051_PART_ADDRESS")
T1096 = src("T1096_PATERNITY")

name_parts = (stg_vdemo
  .withColumn("_nam",   F.trim(F.col("FullDisplay_NAME")))
  .withColumn("_last",  F.trim(F.split(F.col("_nam"), ",").getItem(0)))
  .withColumn("_rest",  F.trim(F.split(F.col("_nam"), ",").getItem(1)))
  .withColumn("_first", F.split(F.col("_rest"), " ").getItem(0))
  .withColumn("_mid",   F.expr("trim(substring(_rest, length(_first)+1, 100))")))

dl = (T1005
  .where((F.col("LIC_AGENCY_CD") == "DMV") & (F.col("LIC_TYPE_CD").isin("D", "DRV")))
  .groupBy("NCP_MCI_NUM")
  .agg(F.first("LIC_NUM", ignorenulls=True).alias("LicenseDriverNo_TEXT")))

mail_phone = (T1051
  .where(F.col("ADD_TYP_CD") == "MAIL")
  .groupBy("MCI_NUM")
  .agg(F.first("HSE_PHONE_NUM", ignorenulls=True).alias("HomePhone_NUMB")))

vdemo = (name_parts.alias("p")
  .join(T1045.alias("x"),  F.col("p.MCI_NUM") == F.col("x.MCI_NUM"), "left")
  .join(T1096.alias("pat"), F.col("p.MCI_NUM") == F.col("pat.CHILD_MCI_NUM"), "left")
  .join(dl, F.col("p.MCI_NUM") == dl["NCP_MCI_NUM"], "left")
  .join(mail_phone, F.col("p.MCI_NUM") == mail_phone["MCI_NUM"], "left")
  .select(
      F.col("p.MemberMci_IDNO"),
      F.col("p.INDIVIDUAL_IDNO"),
      F.col("p._last").alias("Last_NAME"),
      F.col("p._first").alias("First_NAME"),
      F.col("p._mid").alias("Middle_NAME"),
      F.col("p.FullDisplay_NAME"),
      F.col("p.MemberSex_CODE"),
      F.col("p.MemberSsn_NUMB"),
      F.col("p.Birth_DATE"),
      F.trim(F.col("x.BIRTH_CITY")).alias("BirthCity_NAME"),
      F.coalesce(F.col("pat.BIRTH_ST"), F.col("x.BIRTH_STATE")).alias("BirthState_CODE"),
      F.col("p.BirthCountry_CODE"),
      F.regexp_replace(F.col("x.HEIGHT"), " ", "").alias("DescriptionHeight_TEXT"),
      F.col("x.WEIGHT").cast("string").alias("DescriptionWeightLbs_TEXT"),
      F.col("p.Race_CODE"),
      F.trim(F.col("x.HAIR_COLOR")).alias("ColorHair_CODE"),
      F.trim(F.col("x.EYE_COLOR")).alias("ColorEyes_CODE"),
      F.col("p.Language_CODE"),
      F.col("HomePhone_NUMB"),
      F.col("LicenseDriverNo_TEXT"),
      F.col("p.BeginValidity_DATE"),
      F.col("p.WorkerUpdate_ID"),
      F.current_timestamp().alias("Update_DTTM"),
  ))

write_target(conform(vdemo), "VDEMO")

## 3. VCASE  –  add Intercept / RespondInit / Event-derived dates / CaseCategory

In [ ]:
stg_vcase = stg("stg_VCASE_Simple")
T1038 = src("T1038_FTIN_ELIG")
T1074 = src("T1074_EVENT")
T1082 = src("T1082_CASE_PART")
T1051 = src("T1051_PART_ADDRESS")

intercept = (T1038.where(F.col("TAX_INT_STS_CD").isin("URRS", "SUBM"))
                .select("CASE_NUM").distinct()
                .withColumn("_intercept", F.lit("I")))

noncoop  = (T1074.where(F.col("TYP_CD") == "UCNC")
                .groupBy("CASE_NUM").agg(F.min("CREATE_DT").alias("NonCoop_DATE")))

goodcause = (T1074.where(F.col("TYP_CD").isin("WPFA", "UPFA"))
                  .groupBy("CASE_NUM").agg(F.min("CREATE_DT").alias("GoodCause_DATE")))

ncp = (T1082.where(F.col("TYP_CD") == "AP  ")
             .select("CASE_NUM", F.col("MCI_NUM").alias("NCP_MCI")))
ncp_mail = (ncp.alias("n")
  .join(T1051.where(F.col("ADD_TYP_CD") == "MAIL").alias("a"),
        F.col("n.NCP_MCI") == F.col("a.MCI_NUM"), "left")
  .select("n.CASE_NUM",
          F.when(F.col("a.STATE_ADR") == "DE", "I")
           .when(F.col("a.STATE_ADR").isNotNull(), "R")
           .otherwise("I").alias("RespondInit_CODE")))

case_cat = (F.when(F.col("IVD_TYP_CD") == "DIRP", "DP")
             .when(F.col("IVD_TYP_CD") == "MAO ", "MO")
             .when(F.col("IVD_TYP_CD") == "FC  ", "FC")
             .otherwise("IV"))

vcase = (stg_vcase.alias("v")
  .join(intercept,  F.col("v.Case_IDNO") == intercept["CASE_NUM"],  "left")
  .join(noncoop,    F.col("v.Case_IDNO") == noncoop["CASE_NUM"],    "left")
  .join(goodcause,  F.col("v.Case_IDNO") == goodcause["CASE_NUM"],  "left")
  .join(ncp_mail,   F.col("v.Case_IDNO") == ncp_mail["CASE_NUM"],   "left")
  .withColumn("Intercept_CODE",   F.coalesce(F.col("_intercept"), F.lit("N")))
  .withColumn("CaseCategory_CODE", case_cat)
  .withColumn("StatusEnforce_CODE",
              F.when(F.col("v.StatusCase_CODE") == "C", "CLSD").otherwise("ACTV"))
  .drop("_intercept", "CASE_NUM", "STATUS_CD", "IVD_TYP_CD", "CLOSING_CD", "COUNTY_CD", "GOOD_CAUSE_RSN_CD")
)

write_target(conform(vcase), "VCASE")

## 4. VCMEM  –  add CpRel/NcpRel + FAMILYVIOLENCE_INDC

In [ ]:
stg_vcmem = stg("stg_VCMEM_Simple")
T1043 = src("T1043_PART")
T1044 = src("T1044_CASE")

fv = (T1044.where(F.col("GOOD_CAUSE_RSN_CD").isin("WPFA", "UPFA") |
                  F.col("UNWORK_RSN_CD").isin("WPFA", "UPFA"))
          .select("CASE_NUM").distinct()
          .withColumn("_fv", F.lit("Y")))

vcmem = (stg_vcmem.alias("m")
  .join(T1043.alias("p"), F.col("m.MCI_NUM") == F.col("p.MCI_NUM"), "left")
  .join(fv, F.col("m.Case_IDNO") == fv["CASE_NUM"], "left")
  .withColumn("CpRelationshipToChild_CODE",
              F.when(F.col("m.TYP_CD") == "CP  ",
                     F.when(F.col("p.SEX_CD") == "F", "MOT").otherwise("FAT"))
               .when(F.col("m.TYP_CD") == "DEP ", F.lit("MOT")))
  .withColumn("NcpRelationshipToChild_CODE",
              F.when(F.col("m.TYP_CD").isin("AP  ", "DEP "), F.lit("FAT")))
  .withColumn("FAMILYVIOLENCE_INDC", F.coalesce(F.col("_fv"), F.lit("N")))
  .drop("_fv", "CASE_NUM", "TYP_CD")
  .drop(F.col("p.MCI_NUM")))

write_target(conform(vcmem), "VCMEM")

## 5. VSORD  –  add File_ID + IssuingOrderFips_CODE + DirectPay_INDC

In [ ]:
stg_vsord = stg("stg_VSORD_Simple")
T1044 = src("T1044_CASE")

STATE_FIPS = {"DE":"1000300","PA":"4200000","NJ":"3400000","MD":"2400000",
              "NY":"3600000","VA":"5100000","CA":"0600000","TX":"4800000"}
fmap = F.create_map([F.lit(x) for kv in STATE_FIPS.items() for x in kv])

vsord = (stg_vsord.alias("o")
  .join(T1044.alias("c"), F.col("o.Case_IDNO") == F.col("c.CASE_NUM"), "left")
  .withColumn("File_ID", F.regexp_replace(F.col("c.FAM_COURT_FILE"), "-", ""))
  .withColumn("IssuingOrderFips_CODE",
              F.coalesce(fmap.getItem(F.col("o.ORIGINATING_ST_CD")), F.lit(DELAWARE_FIPS)))
  .withColumn("DirectPay_INDC",
              F.when(F.col("c.IVD_TYP_CD") == "DIRP", "Y").otherwise("N"))
  .withColumn("Order_IDNO",
              F.lit(900000000000000) +
              (F.col("o.Case_IDNO").cast("long") * 100 + F.col("o.OrderSeq_NUMB")))
  .drop("CASE_NUM", "FAM_COURT_FILE", "ORIGINATING_ST_CD", "MOD_TYP_CD", "IVD_TYP_CD"))

write_target(conform(vsord), "VSORD")

## 6. VOBLE  –  full notebook responsibility (no DF stage)

Skip DIRP cases, build composite Fips_CODE, window for ObligationSeq_NUMB, conditional Periodic_AMNT.

In [ ]:
T1049 = src("T1049_COURT_ORDER")
T1050 = src("T1050_ORD_SUB_ACCT")
T1048 = src("T1048_SUB_ACCT")
T1044 = src("T1044_CASE")
T1051 = src("T1051_PART_ADDRESS")
T1082 = src("T1082_CASE_PART")
ref   = src("t_reference")

non_dirp = T1044.where(F.col("IVD_TYP_CD") != "DIRP").select("CASE_NUM").distinct()

cp = T1082.where(F.col("TYP_CD") == "CP  ").select("CASE_NUM", F.col("MCI_NUM").alias("CP_MCI"))
cp_mail = (cp.alias("c").join(T1051.where(F.col("ADD_TYP_CD") == "MAIL").alias("a"),
                              F.col("c.CP_MCI") == F.col("a.MCI_NUM"), "left")
             .select("c.CASE_NUM",
                     F.lpad(F.col("a.PAYMENT_FIPS_CD").cast("string"), 7, "0").alias("_mail_fips")))

ostata = (T1048.where(F.col("SUB_TYP_CD") == "OSTATA")
                .groupBy("CASE_NUM").agg(F.first("URESA_CD", ignorenulls=True).alias("_ost")))

td = ref.where((F.col("view_nam")=="VOBLE") & (F.col("field_nam")=="TypeDebt_CODE")) \
        .select(F.col("old_code").alias("_o"), F.col("new_code").alias("TypeDebt_CODE"))
fp = ref.where((F.col("view_nam")=="VOBLE") & (F.col("field_nam")=="FreqPeriodic_CODE")) \
        .select(F.col("old_code").alias("_o2"), F.col("new_code").alias("FreqPeriodic_CODE"))

voble = (T1050.alias("l")
  .join(non_dirp, "CASE_NUM", "inner")
  .join(T1049.alias("o"),
        (F.col("l.CASE_NUM")==F.col("o.CASE_NUM")) & (F.col("l.BLIND_KEY")==F.col("o.BLIND_KEY")), "left")
  .join(T1044.alias("c"), F.col("l.CASE_NUM")==F.col("c.CASE_NUM"), "left")
  .join(cp.alias("cp"),   F.col("l.CASE_NUM")==F.col("cp.CASE_NUM"), "left")
  .join(cp_mail,          F.col("l.CASE_NUM")==cp_mail["CASE_NUM"], "left")
  .join(ostata,           F.col("l.CASE_NUM")==ostata["CASE_NUM"], "left")
  .join(td, F.rtrim(F.col("l.SUB_TYP_CD"))==F.rtrim(F.col("_o")),  "left")
  .join(fp, F.rtrim(F.col("o.CHRG_FREQ_CD"))==F.rtrim(F.col("_o2")), "left"))

w = Window.partitionBy("l.CASE_NUM", "l.BLIND_KEY").orderBy("l.SUB_TYP_CD")
voble = voble.withColumn("ObligationSeq_NUMB", F.row_number().over(w))

fips_rule = (F.when(F.col("l.SUB_TYP_CD")=="OSTATA",
                    F.lpad(F.col("_ost").cast("string"), 7, "0"))
              .when(F.col("_mail_fips").isNotNull() & (F.col("_mail_fips")!="0000000"),
                    F.col("_mail_fips"))
              .otherwise(F.lit(DELAWARE_FIPS)))

periodic = (F.when(F.col("l.SUB_TYP_CD").isin("SPLSUP","CURSUP","MEDSUP","CSCHLD"),
                   F.col("l.FREQ_AMT"))
             .otherwise(F.lit(0).cast(DecimalType(11,2))))

voble_out = voble.select(
    F.col("l.CASE_NUM").cast("long").alias("Case_IDNO"),
    F.col("l.BLIND_KEY").cast("int").alias("OrderSeq_NUMB"),
    F.col("ObligationSeq_NUMB"),
    (F.col("cp.CP_MCI").cast("long") + F.lit(1000000)).alias("MemberMci_IDNO"),
    F.coalesce(F.col("TypeDebt_CODE"), F.lit("OT")).alias("TypeDebt_CODE"),
    fips_rule.alias("Fips_CODE"),
    F.coalesce(F.col("FreqPeriodic_CODE"), F.lit("M")).alias("FreqPeriodic_CODE"),
    periodic.alias("Periodic_AMNT"),
    periodic.alias("ExpectToPay_AMNT"),
    F.lit("C").alias("ExpectToPay_CODE"),
    F.col("o.ORDER_START_DT").alias("BeginObligation_DATE"),
    F.col("o.ORDER_END_DT").alias("EndObligation_DATE"),
    F.col("c.LAST_CHRG_DT").alias("AccrualLast_DATE"),
    F.col("c.NEXT_CHARGE_DT").alias("AccrualNext_DATE"),
    F.to_date(F.lit(CONVERSION_DATE)).alias("BeginValidity_DATE"),
)

write_target(conform(voble_out), "VOBLE")

## 7. VLSUP  –  full notebook responsibility (aggregations on VOBLE)

In [ ]:
T1018 = src("T1018_SS_DEBT")
voble = stg("VOBLE")

b18 = T1018.groupBy("CASE_NUM").agg(
    F.sum(F.when(F.rtrim(F.col("SS_TYP_CD"))=="PERMAN", F.col("BAL_DUE")).otherwise(0)).alias("OweTotPaa_AMNT"),
    F.sum(F.when(F.rtrim(F.col("SS_TYP_CD"))=="UNASSD", F.col("BAL_DUE")).otherwise(0)).alias("OweTotUda_AMNT"),
    F.sum(F.when(~F.rtrim(F.col("SS_TYP_CD")).isin("PERMAN","UNASSD"), F.col("BAL_DUE")).otherwise(0)).alias("OweTotNaa_AMNT"),
    F.sum("BAL_DUE").alias("OweTotCurSup_AMNT"))

b48 = src("T1048_SUB_ACCT").groupBy("CASE_NUM").agg(
    F.sum(F.when(F.rtrim(F.col("SUB_TYP_CD"))=="FCAREA", F.col("BALANCE").cast(DecimalType(13,2))).otherwise(0)).alias("OweTotIvef_AMNT"),
    F.sum(F.when(F.rtrim(F.col("SUB_TYP_CD"))=="MEDA",   F.col("BALANCE").cast(DecimalType(13,2))).otherwise(0)).alias("OweTotMedi_AMNT"))

tw = src("T1044_CASE").select("CASE_NUM",
        F.when(F.col("IVD_TYP_CD")=="AFDC","A")
         .when(F.col("IVD_TYP_CD")=="MAO ","M")
         .otherwise("N").alias("TypeWelfare_CODE"))

vlsup = (voble.alias("v")
  .join(b18, F.col("v.Case_IDNO")==b18["CASE_NUM"], "left")
  .join(b48, F.col("v.Case_IDNO")==b48["CASE_NUM"], "left")
  .join(tw,  F.col("v.Case_IDNO")==tw["CASE_NUM"],  "left")
  .select("v.Case_IDNO", "v.OrderSeq_NUMB", "v.ObligationSeq_NUMB",
          F.date_format(F.current_date(), "yyyyMM").cast("int").alias("SupportYearMonth_NUMB"),
          "TypeWelfare_CODE",
          F.col("v.Periodic_AMNT").alias("TransactionCurSup_AMNT"),
          F.coalesce(F.col("OweTotCurSup_AMNT"), F.lit(0)).alias("OweTotCurSup_AMNT"),
          F.coalesce(F.col("OweTotPaa_AMNT"),    F.lit(0)).alias("OweTotPaa_AMNT"),
          F.coalesce(F.col("OweTotUda_AMNT"),    F.lit(0)).alias("OweTotUda_AMNT"),
          F.coalesce(F.col("OweTotNaa_AMNT"),    F.lit(0)).alias("OweTotNaa_AMNT"),
          F.coalesce(F.col("OweTotIvef_AMNT"),   F.lit(0)).alias("OweTotIvef_AMNT"),
          F.coalesce(F.col("OweTotMedi_AMNT"),   F.lit(0)).alias("OweTotMedi_AMNT"),
          F.to_date(F.lit(CONVERSION_DATE)).alias("Batch_DATE"),
          F.lit("WGE").alias("SourceBatch_CODE")))

write_target(conform(vlsup), "VLSUP")

## 8. Audit table for the pipeline `Validate_Row_Counts` activity

In [ ]:
rows = []
for t in ["VDEMO","VCASE","VCMEM","VSORD","VOBLE","VLSUP"]:
    rows.append((t, spark.read.table(f"{TARGET_LAKEHOUSE}.{t}").count()))
audit = spark.createDataFrame(rows, ["table_name", "row_count"]) \
             .withColumn("audit_dttm", F.current_timestamp())
write_target(audit, "v_conversion_audit")

## 9. Return to pipeline

Exits the notebook with a JSON payload of row counts so the pipeline activity output (ctivity("Run_Hybrid_Notebook").output.result.exitValue) can drive downstream branching or be logged.

In [ ]:
import json

# notebookutils is auto-injected by the Fabric Spark runtime.
payload = {
    "status": "ok",
    "conversion_date": CONVERSION_DATE,
    "row_counts": ROW_COUNTS,
}
notebookutils.notebook.exit(json.dumps(payload))
